# Lab 08: ADK Graph Workflows and n8n Coexistence
Building Enterprise AI Agents on **Gemini Enterprise Agent Platform (GEAP)**: Pulse Telco Workshop

---

## 🎯 Objectives
In Labs 01 to 07 you built **Pulse Concierge**, an open-ended conversational agent where the language model decides at runtime which tools and sub-agents to invoke next. That autonomy works well for exploratory customer care and executive analytics, where questions arrive in unpredictable shapes.

Network operations and billing teams also run **pre-defined business processes**, often in visual workflow tools such as **n8n** today, where the execution path must be fixed, deterministic, and auditable. When a 5G cell site drops packets, the operations center cannot leave it up to a model's sampling temperature whether to check subscriber churn exposure or pause for a shift lead's sign-off before dispatching a field crew.

In this lab you build **pulse-workflow-agent**, a 5G incident remediation workflow implemented with the **ADK 2.x graph Workflow API**, and compare it directly with n8n across architecture and runtime mechanics:
1. **Normalize multi-channel triggers**: Accept chat text, a voice request, or a structured JSON event (such as a Pub/Sub push notification from a NOC alert or a Cloud Scheduler cron job) and coerce all of them into a single validated Pydantic schema.
2. **Fan out in parallel**: Run a deterministic Python node that queries tower telemetry from BigQuery concurrently with an LLM agent node that evaluates CRM and billing impact through the BigQuery MCP server, the Knowledge Catalog MCP server, and Agent Registry skills.
3. **Fan in and route by risk**: Synchronize both branches with *JoinNode*, then evaluate severity in code so every P1 outage requires a human decision regardless of which channel started the run, while lower-severity incidents take a monitor-only branch.
4. **Pause and resume across human approval**: Yield a *RequestInput* checkpoint that pauses execution until an operator resumes the run with **APPROVED** (triggering field dispatch and SLA retention credits) or **REJECTED** (canceling dispatch and preserving the operator's rationale).
5. **Drive the same graph from four entry points**: Start the workflow interactively from **chat**, through a **voice front door** (text prompt in, spoken audio answer out), and through ADK's built-in **Pub/Sub trigger route** running locally.
6. *(Optional)* **Deploy to Agent Runtime**: Package and deploy the workflow as a managed cloud service with its own dedicated Agent Identity.

**Where this fits in the development lifecycle:** In Labs 03 to 06 we iterated on Pulse Concierge locally against managed cloud services and deployed the final container once at the end of Lab 06. This lab scaffolds a **separate** agent project in *pulse-workflow-agent/* so its dependencies and deployment lifecycle remain decoupled from the conversational concierge.

### Key references
* 🕸️ [ADK Workflows: multi-agent, multi-node applications](https://adk.dev/workflows/)
* 🔀 [ADK graph routes (parallel fan-out and JoinNode)](https://adk.dev/graphs/routes/)
* ✋ [ADK human input (RequestInput)](https://adk.dev/graphs/human-input/)
* ⚡ [ADK ambient agents: Pub/Sub and Eventarc trigger endpoints](https://adk.dev/runtime/ambient-agents/)
* 🎙️ [ADK live and voice agents (marked experimental)](https://adk.dev/live/)
* 🔌 [ADK Application Integration tool (Integration Connectors)](https://adk.dev/integrations/application-integration/)

---

## 🗺️ Architecture: the *pulse_incident_workflow* graph

```mermaid
flowchart TD
    T1["💬 Chat: run_async"] --> START(("START"))
    T2["🎙️ Voice front door: run_live calls a function tool"] --> START
    T3["⚡ Pub/Sub push: NOC alert -> /trigger/pubsub"] --> START
    T4["⏰ Cloud Scheduler -> Pub/Sub -> same route"] --> START
    START --> P["1. parse_trigger_payload (@node)"]
    P --> B1["2a. tower_telemetry_node (@node)<br/>BigQuery telco_network, no LLM"]
    P --> B2["2b. crm_billing_impact_agent (LlmAgent)<br/>BigQuery MCP + Knowledge Catalog MCP + skills"]
    B1 --> J["3. merge_diagnostics (JoinNode)"]
    B2 --> J
    J --> R{"4. evaluate_severity_router<br/>P1 outage?"}
    R -- "require_hitl_approval (any channel)" --> H["5. hitl_dispatch_approval_node<br/>RequestInput pauses the run"]
    R -- "monitor_only" --> M["monitor_only_node<br/>MONITORING"]
    H -- "approved" --> E["execute_remediation_node<br/>REMEDIATION_DISPATCHED"]
    H -- "rejected" --> X["log_rejection_node<br/>DISPATCH_REJECTED"]
```

## 💶 Cost and Scaling

| Service | How it is billed (list price, verified 2026-10-20) | This lab |
| :--- | :--- | :--- |
| Second Agent Runtime instance (*europe-west4*, optional deploy) | **\$0.085/vCPU-h + \$0.009/GiB-h** while processing (**\$0.376/h** at default 4 vCPU / 4 GiB). Idle instances and idle minimum instances are not billed (Preview; subject to change). Plus Cloud Build (\$0.006/build-min, 2,500 free) and Artifact Registry (\$0.10/GiB-month, 0.5 GiB free) | ≈ **\$0.01** (or \$0.00 if skipped) |
| Gemini 3.8 Flash on *eu* (*crm_billing_impact_agent* node) | **\$1.65** per 1M input tokens and **\$8.25** per 1M output tokens (*eu* multi-region carries a 10% regional premium vs *global*; 50% promotional credit applies through 2026-12-31) | ~4 workflow runs ≈ **\$0.25** |
| Gemini 3.8 Live on *eu* (*voice_workflow_dispatcher*) | Audio input **\$3.00**, audio output **\$12.00**, text input **\$0.75**, text output **\$4.50** per 1M tokens (25 tokens/s of audio; prior turns re-billed per turn) | 1 voice turn ≈ **\$0.02** |
| BigQuery (*EU*), Knowledge Catalog (*eu*), Skill Registry (*eu*), Agent Registry | BigQuery on-demand **\$6.25/TiB** (first 1 TiB/month free). Knowledge Catalog API and search **free**. Skill Registry reads billed at 1 vCPU-h per 3M calls (Preview). Agent Registry and managed MCP layer carry **no separate public fee** | ≈ **\$0.00** |
| Pub/Sub and Cloud Scheduler (production trigger architecture) | Pub/Sub: **\$40/TiB**, first **10 GiB/month free**. Cloud Scheduler: **\$0.10 per job per 31 days**, **3 free jobs per billing account** | **\$0.00** (in-process *TestClient* in this lab) |

**Scaling characteristics and quotas:**
* **Agent Runtime (*europe-west4*):** Supports *min_instances* from 0 to 10 (default 1) and *max_instances* from 1 to 1000 (default 100), with a maximum synchronous *streamQuery* duration of **10 minutes** (long-running background jobs support up to 7 days). Default regional quotas per project: **100 Agent Runtime instances**, **10 create/update/delete operations per minute**, and **90 queries per minute**.
* **ADK trigger endpoints:** *ADK_TRIGGER_MAX_CONCURRENT* (default **10**) bounds concurrent Pub/Sub push executions per container instance, while *ADK_TRIGGER_MAX_RETRIES* (default **3**) retries transient HTTP 429 responses with exponential backoff. Cloud Pub/Sub scales to 4 GB/s per region, and Cloud Scheduler supports up to 5,000 jobs per region.

**Estimated cost for this lab:** **≈ \$0.30** at list price (≈ **\$0.15** after the 50% Gemini 3.8 Flash promotional credit).

*Sources: [GEAP pricing](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing) · [Generative AI pricing](https://cloud.google.com/gemini-enterprise-agent-platform/generative-ai/pricing) · [Agent quotas](https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/agent-quotas) · [Pub/Sub pricing](https://cloud.google.com/pubsub/pricing) · [Cloud Scheduler pricing](https://cloud.google.com/scheduler/pricing).*

## ⚖️ ADK Graph Workflows vs n8n: Architectural Trade-Offs and Coexistence

A common mistake in enterprise architecture is treating a new platform as a mandate to rewrite every working workflow. Visual automation engines like **n8n** and code-first graph frameworks like **ADK 2.x Workflow** optimize for different parts of the organization. n8n gives operations and business teams a visual canvas and hundreds of turnkey SaaS integrations. ADK gives software engineers typed Python graphs where LLM agents, MCP servers, Model Armor guardrails, and automated evaluations are native building blocks.

The table below compares current n8n capabilities (2025 to 2026) with ADK 2.x *Workflow* graphs on GEAP (sources linked at the end of this section).

| Dimension | n8n (2025 to 2026) | ADK 2.x *Workflow* on GEAP | Architectural fit |
| :--- | :--- | :--- | :--- |
| **Authoring model** | Visual canvas, JSON export, Git-based source control and environments (Business and Enterprise plans). Operations and business teams can own flows directly | Python + Pydantic schemas: standard PR diffs, unit tests, and typed node contracts. Requires software engineering skills | n8n for citizen and ops automation; ADK for engineered, LLM-intensive workflows |
| **SaaS connectors** | Extensive catalog of turnkey nodes and webhook triggers (Teams, Outlook, SharePoint, Jira, ServiceNow) | MCP servers, ADK toolsets, and Google Cloud Integration Connectors via *ApplicationIntegrationToolset*. More setup per connector | **n8n** |
| **AI agent depth** | AI Agent node (LangChain-based), memory, vector stores, MCP Client Tool, and MCP Server Trigger | LLM agents as native graph nodes with Skill Registry, Memory Bank, Model Armor screening, and managed evaluation (Labs 03 to 07) | ADK for governance and evaluation depth; n8n for rapid prototyping |
| **Parallel execution** | In workflows created with n8n 1.0+, multiple branches wired to one node execute sequentially (one branch completes before the next starts) | True concurrent fan-out via tuple edges plus typed synchronization via *JoinNode* | **ADK** |
| **Human-in-the-loop approval** | Wait node (execution state parked in the n8n database) plus turnkey "Send and Wait for Response" actions in Teams, Outlook, Slack, and Gmail | *RequestInput* checkpoint and API-driven resume with typed state in managed Sessions. You provide the approval UI (Teams card or web form) | n8n for turnkey approval UX; ADK for durable typed state |
| **Identity and audit** | Shared credentials stored inside the n8n credential store (OAuth2, static API keys) across runs of the workflow | Dedicated **Agent Identity** principal per deployed agent, governed by GCP IAM and recorded in Cloud Audit Logs | **ADK / GEAP** |
| **Operations and scale** | Self-hosted queue mode (Redis + worker fleet + PostgreSQL) or n8n Cloud; visual execution log UI with single-node retry | Managed Agent Runtime autoscaling, Cloud Trace, Cloud Logging, and managed sessions; no visual canvas replay UI | Depends on whether your platform team runs Kubernetes/Redis or serverless GCP |
| **Quality evaluation** | No built-in automated evaluation harness for LLM trajectory or rubric grading | Local, managed, and continuous online evaluation (Lab 07) | **ADK / GEAP** |
| **Cost structure** | Self-hosted infrastructure + enterprise license, or cloud tiers billed per workflow execution | Agent Runtime vCPU/GiB-hours while active + Gemini token consumption + underlying GCP service usage | Compare against your execution frequency and token volume |

**What stays in n8n:** Connector-heavy SaaS glue workflows with minimal LLM reasoning that are already maintained by operations or business teams. Rewriting stable point-to-point automations into Python adds maintenance burden without architectural upside.

**How they coexist in production:** Rather than choosing one exclusively, connect them across a clean protocol boundary. An n8n workflow can invoke a deployed GEAP agent using its standard HTTP Request node (calling either the Agent Runtime query endpoint or the agent's A2A endpoint) and handle the downstream Microsoft Teams approval card or Jira ticket creation. Conversely, a GEAP agent can invoke n8n workflows that n8n exposes through its **MCP Server Trigger** node, consuming them through the exact same *McpToolset* pattern you used in Lab 03.

```mermaid
flowchart LR
    subgraph N8N["n8n"]
        NF["Connector-heavy flow<br/>(Teams, Outlook, Jira)"]
        MST["MCP Server Trigger<br/>(n8n workflows as MCP tools)"]
    end
    subgraph GEAP["GEAP"]
        AG["ADK agent or workflow<br/>(Agent Runtime, own Agent Identity)"]
    end
    NF -- "HTTP Request: query API or A2A" --> AG
    AG -- "McpToolset" --> MST
```

### Choosing Between Ready-Made Connectors and Custom Code
When an ADK workflow needs to reach an external enterprise system, evaluate three integration options in order of engineering effort:
1. **Existing n8n workflows exposed via MCP**: Fastest path when your team already maintains the SaaS credentials and flow logic inside n8n.
2. **Google Cloud Integration Connectors via ADK's *ApplicationIntegrationToolset***: Managed enterprise connectors covering Microsoft Teams, Office 365, SharePoint, Jira Cloud, Confluence, ServiceNow, Salesforce, and SAP. You provision a connection once in the project and expose only the specific entity operations or actions the agent should use:

```python
jira_tools = ApplicationIntegrationToolset(
    project=PROJECT_ID,
    location=REGION,
    connection="jira-conn",
    entity_operations={"Issues": ["LIST", "CREATE"]},
)
```

3. **Custom MCP servers on Cloud Run**: The pattern you built for the IAM Context MCP server in Lab 03. It gives you full control over request validation, tool annotations, and internal API calls when no off-the-shelf connector exists.

In practice, teams combine all three: Integration Connectors for standard enterprise SaaS, custom MCP servers for proprietary internal systems, and n8n for high-touch cross-app workflows that operations teams already own.

Sources: n8n [execution order](https://docs.n8n.io/build/flow-logic/understand-execution-order) · [Microsoft Teams node (Send and Wait for Response)](https://docs.n8n.io/integrations/builtin/app-nodes/n8n-nodes-base.microsoftteams) · [human-in-the-loop for tools](https://docs.n8n.io/build/integrate-ai/ai-examples/human-in-the-loop-for-tools) · [Wait node](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait) · [MCP Server Trigger](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-langchain.mcptrigger) · [MCP Client Tool](https://docs.n8n.io/integrations/builtin/cluster-nodes/sub-nodes/n8n-nodes-langchain.toolmcp) · [queue mode](https://docs.n8n.io/deploy/host-n8n/configure-n8n/scaling/enable-queue-mode) · [source control and environments](https://docs.n8n.io/administer/use-source-control-and-environments) · [pricing](https://n8n.io/pricing/). Google: [All Integration Connectors](https://docs.cloud.google.com/integration-connectors/docs/all-integration-connectors).

## Step 8.0: Load Workshop Configuration

We load *workshop.env* (which holds the BigQuery dataset names from Lab 00 along with the IAM Context MCP resource name and Skill Registry location from Lab 03) and define three notebook helpers:
* *save_to_workshop_env()*: persists key-value pairs to *workshop.env* and the active kernel environment for subsequent labs.
* *show_event()* and *show_stats()*: inspect every tool call, tool response, error, and model message emitted during both local runner executions and remote Agent Runtime streams.

In [ ]:
# Step 8.0: Load workshop configuration and define event-printing helpers
import os, sys, json, pathlib, subprocess
from dotenv import load_dotenv

os.environ.setdefault("GOOGLE_API_USE_CLIENT_CERTIFICATE", "false")

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID            = os.environ["PROJECT_ID"]
PROJECT_NUMBER        = os.environ["PROJECT_NUMBER"]
REGION                = os.environ.get("REGION", "europe-west4")          # workshop region: Agent Runtime, Agent Registry, Cloud Run
MODEL_LOCATION        = os.environ.get("MODEL_LOCATION", "eu")            # eu: multi-region endpoint for the Gemini models
LIVE_LOCATION         = os.environ.get("LIVE_LOCATION", "eu")             # eu: multi-region endpoint for the Gemini Live model
PULSE_SKILLS_LOCATION = os.environ.get("PULSE_SKILLS_LOCATION", "eu")     # eu: multi-region Skill Registry
MODEL                 = os.environ.get("MODEL", "gemini-3.8-flash")       # gemini-3.8-flash
BQ_DATASET_CRM        = os.environ.get("BQ_DATASET_CRM", "telco_crm")
BQ_DATASET_BILLING    = os.environ.get("BQ_DATASET_BILLING", "telco_billing")
BQ_DATASET_NETWORK    = os.environ.get("BQ_DATASET_NETWORK", "telco_network")
IAM_MCP_RESOURCE      = os.environ["PULSE_IAM_MCP_RESOURCE"]
WF_AGENT_NAME         = "pulse-workflow-agent"
WF_AGENT_DIR          = REPO_ROOT / WF_AGENT_NAME

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "REGION": REGION,
    "MODEL_LOCATION": MODEL_LOCATION,
    "LIVE_LOCATION": LIVE_LOCATION,
    "PULSE_LIVE_LOCATION": LIVE_LOCATION,
    "PULSE_SKILLS_LOCATION": PULSE_SKILLS_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "GOOGLE_API_USE_CLIENT_CERTIFICATE": "false",
    "UV_DEFAULT_INDEX": "https://pypi.org/simple",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION} | live: {LIVE_LOCATION} | skills: {PULSE_SKILLS_LOCATION}")
print(f"Workflow agent dir: {WF_AGENT_DIR} | IAM MCP: {IAM_MCP_RESOURCE.split('/')[-1]}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")

def save_to_workshop_env(**kv):
    """Persist values for the next labs (workshop.env, KEY=VALUE lines) and for this kernel."""
    lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
    for k, v in kv.items():
        lines = [l for l in lines if not l.startswith((f"{k}=", f"export {k}="))]
        lines.append(f"{k}={v}")
        os.environ[k] = str(v)
    ENV_FILE.write_text("\n".join(lines) + "\n")
    print("saved:", ", ".join(f"{k}={v}" for k, v in kv.items()))

import json


def show_event(ev, stats=None, text_limit=2000):
    """Print one agent event: tool calls, tool results, errors and text.

    Works for local ADK `Event` objects (Runner.run_async / run_live) and for the
    dicts a deployed agent streams (async_stream_query), which use snake_case keys.
    Pass a dict as `stats` to accumulate token usage and the number of model calls.
    """
    d = ev if isinstance(ev, dict) else ev.model_dump(exclude_none=True)
    who = d.get("author", "agent")
    err_code = d.get("error_code") or d.get("errorCode")
    err_msg = d.get("error_message") or d.get("errorMessage")
    if err_code or err_msg:
        print(f"  !! [{who}] ERROR {err_code or ''}: {str(err_msg or '')[:400]}")
    for part in (d.get("content") or {}).get("parts") or []:
        fc = part.get("function_call") or part.get("functionCall")
        fr = part.get("function_response") or part.get("functionResponse")
        if fc:
            args = json.dumps(fc.get("args") or {}, ensure_ascii=False, default=str)
            print(f"  -> [{who}] tool call   {fc.get('name')}({args[:300]})")
        elif fr:
            resp = json.dumps(fr.get("response") or {}, ensure_ascii=False, default=str)
            print(f"  <- [{who}] tool result {fr.get('name')}: {resp[:300]}")
        elif part.get("text") and not part.get("thought"):
            print(f"  [{who}] {part['text'].strip()[:text_limit]}")
    usage = d.get("usage_metadata") or d.get("usageMetadata")
    if stats is not None and usage:
        stats["model_calls"] = stats.get("model_calls", 0) + 1
        for key in ("prompt_token_count", "candidates_token_count", "thoughts_token_count"):
            stats[key] = stats.get(key, 0) + (usage.get(key) or usage.get(key.replace("_token_count", "TokenCount")) or 0)


def show_stats(stats):
    if stats:
        print(f"  (model calls: {stats.get('model_calls', 0)} | tokens in: {stats.get('prompt_token_count', 0):,} "
              f"| out: {stats.get('candidates_token_count', 0):,} | thinking: {stats.get('thoughts_token_count', 0):,})")

## Step 8.1: Scaffold the Workflow Project, Write Environment Settings, and Pin ADK 2.11.0

To keep the deterministic incident workflow independent from the conversational concierge, we scaffold a dedicated ADK project in *pulse-workflow-agent/*. Separating the two directories ensures each agent maintains its own *pyproject.toml*, container build context, and *deployment_metadata.json*.

The cell below performs four setup tasks:
* Copies the local *bigquery-basics* skill directory from *pulse-concierge/app/skills/* so the workflow's LLM branch can combine local SQL syntax guidance with the remote *pulse-telco-analytics* skill from Skill Registry.
* Writes *pulse-workflow-agent/.env*, which *agents-cli deploy* bundles into the container environment. Two variables handle subtle differences between local and cloud execution:
  * *PULSE_DATA_PROJECT*: Inside Agent Runtime, the platform automatically sets *GOOGLE_CLOUD_PROJECT* to the numeric **project number**, whereas BigQuery SQL table identifiers require the alphanumeric **project ID**. Storing the project ID in *PULSE_DATA_PROJECT* prevents table-not-found errors after deployment.
  * *PULSE_SKILLS_LOCATION*: Points *GCPSkillRegistry* at the *eu* multi-region endpoint configured in Lab 03.
* Pins **google-adk==2.11.0** and **google-cloud-aiplatform<2.0.0,>=1.156.0** across both the primary dependencies and the *[eval]* optional dependency group in *pyproject.toml*, configures the public PyPI index, adds the MCP and BigQuery client libraries, and runs *uv lock && uv sync*.

⏱ 1 to 3 minutes on the first run.

Equivalent terminal commands:
```bash
agents-cli create pulse-workflow-agent --agent adk --deployment-target agent_runtime \
  --region "$REGION" --prototype --agent-guidance-filename AGENTS.md --skip-checks --yes
cd pulse-workflow-agent
sed -i -E 's/"google-adk\[([^]]*)\][^"]*"/"google-adk[\1]==2.11.0"/' pyproject.toml
sed -i -E 's/"google-cloud-aiplatform\[([^]]*)\][^"]*"/"google-cloud-aiplatform[\1]<2.0.0,>=1.156.0"/' pyproject.toml
uv add "mcp>=1.24,<2" "google-adk[gcp,otel-gcp,agent-identity]==2.11.0" "google-cloud-bigquery>=3.25.0"
uv lock && uv sync
```

In [ ]:
# Step 8.1: Scaffold pulse-workflow-agent, copy the local skill, write .env, and pin ADK 2.11.0
import re, shutil

if not (WF_AGENT_DIR / "agents-cli-manifest.yaml").exists():
    cmd = (f"agents-cli create {WF_AGENT_NAME} --agent adk --deployment-target agent_runtime "
           f"--region {REGION} --prototype --agent-guidance-filename AGENTS.md --skip-checks --yes")
    terminal(cmd)
    subprocess.run(cmd, cwd=REPO_ROOT, shell=True, check=True)
else:
    print(f"{WF_AGENT_DIR} already scaffolded.")

# Local bigquery-basics skill (SkillToolset combines it with the private skill from Agent Registry)
src_skill = REPO_ROOT / "pulse-concierge" / "app" / "skills" / "bigquery-basics"
dst_skill = WF_AGENT_DIR / "app" / "skills" / "bigquery-basics"
if src_skill.exists():
    shutil.rmtree(dst_skill, ignore_errors=True)
    shutil.copytree(src_skill, dst_skill)

# .env = runtime settings for local runs and for the deployed container
env_vars = {
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "REGION": REGION,
    "MODEL_LOCATION": MODEL_LOCATION,
    "LIVE_LOCATION": LIVE_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "1",
    "PULSE_DATA_PROJECT": PROJECT_ID,
    "BQ_LOCATION": os.environ.get("BQ_LOCATION", "EU"),
    "BQ_DATASET_CRM": BQ_DATASET_CRM,
    "BQ_DATASET_BILLING": BQ_DATASET_BILLING,
    "BQ_DATASET_NETWORK": BQ_DATASET_NETWORK,
    "PULSE_REGISTRY_LOCATION": REGION,
    "PULSE_SKILLS_LOCATION": PULSE_SKILLS_LOCATION,
    "PULSE_LIVE_LOCATION": LIVE_LOCATION,
    "PULSE_IAM_MCP_RESOURCE": IAM_MCP_RESOURCE,
    "PULSE_IAM_MCP_URL": os.environ.get("PULSE_IAM_MCP_URL", f"https://pulse-iam-context-mcp-{PROJECT_NUMBER}.{REGION}.run.app/mcp"),
    "GOOGLE_API_USE_CLIENT_CERTIFICATE": "false",
    "GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES": "false",
    "UV_NO_SYNC": "1",
    "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "EVENT_ONLY",
    "ADK_CAPTURE_MESSAGE_CONTENT_IN_SPANS": "true",
}
(WF_AGENT_DIR / ".env").write_text("\n".join(f"{k}={v}" for k, v in env_vars.items()) + "\n")

# Pin google-adk[...] to 2.11.0 and google-cloud-aiplatform[...] to <2.0.0,>=1.156.0 (main + [eval] optional deps),
# ensure public PyPI index, then add libraries and run uv lock && uv sync
pyproject = WF_AGENT_DIR / "pyproject.toml"
ptxt = pyproject.read_text()
ptxt = re.sub(r'"google-adk\[([^\]]*)\][^"]*"', lambda m: f'"google-adk[{m.group(1)}]==2.11.0"', ptxt)
ptxt = re.sub(r'"google-cloud-aiplatform\[([^\]]*)\][^"]*"', lambda m: f'"google-cloud-aiplatform[{m.group(1)}]<2.0.0,>=1.156.0"', ptxt)
if "https://pypi.org/simple" not in ptxt:
    ptxt += '\n[[tool.uv.index]]\nurl = "https://pypi.org/simple"\ndefault = true\n'
pyproject.write_text(ptxt)

cmd = 'uv add "mcp>=1.24,<2" "google-adk[gcp,otel-gcp,agent-identity]==2.11.0" "google-cloud-bigquery>=3.25.0" && uv lock && uv sync'
terminal(cmd, cwd=WF_AGENT_DIR)
r = subprocess.run(cmd, shell=True, cwd=WF_AGENT_DIR, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
if r.returncode != 0:
    raise RuntimeError("uv add / lock / sync failed - see the output above")
print("pulse-workflow-agent is ready:", [l.strip() for l in pyproject.read_text().splitlines() if "google-adk[" in l or "google-cloud-aiplatform[" in l])

## Step 8.2: Author the Typed Incident Graph in ADK 2.x

Why combine deterministic Python functions and LLM agents inside a single graph? Using a language model to fetch a known cell tower's packet-loss percentage wastes tokens and introduces unnecessary latency when a parameterized SQL query can return the exact row in milliseconds. Conversely, synthesizing regional subscriber churn exposure and delinquent billing totals across multiple schemas benefits from an LLM agent equipped with Knowledge Catalog definitions and domain skills.

Here is how the nodes in *pulse-workflow-agent/app/agent.py* fit together at runtime:
1. **Trigger normalization (*parse_trigger_payload*)**: Decorated with *@node*, this function enforces a single Pydantic input schema (*IncidentRequest*) across every inbound channel. Its validator accepts a Python object, a JSON string, free-text chat, or the decoded Pub/Sub push envelope.
2. **Concurrent diagnostic branches (*tower_telemetry_node* and *crm_billing_impact_agent*)**: Wiring a tuple of target nodes to a predecessor edge tells ADK to schedule both branches in parallel:
   * *tower_telemetry_node* is a pure Python *@node* function that executes parameterized SQL against *telco_network* in BigQuery without calling an LLM. Locally it authenticates with your Application Default Credentials; in Agent Runtime it authenticates with the workflow's Agent Identity.
   * *crm_billing_impact_agent* is an *LlmAgent* (constrained by *output_schema=CrmBillingImpact*) that uses the **BigQuery MCP server**, the **Knowledge Catalog MCP server**, the **IAM Context MCP server** (resolved via Agent Registry), and **Skill Registry skills** to calculate active subscribers, High Churn Risk subscribers, and delinquent or disputed EUR invoice exposure for the affected region.
3. **Synchronization barrier (*merge_diagnostics*)**: Uses *JoinNode* to wait until both parallel branches finish, then emits a dictionary keyed by predecessor node name so the downstream router can inspect both diagnostic payloads together.
4. **Policy router (*evaluate_severity_router*)**: Expresses operational risk policy as testable Python code. Every P1 outage emits the *require_hitl_approval* route **regardless of which channel triggered the run**, while non-P1 incidents emit the *monitor_only* route.
5. **Human-in-the-loop checkpoint (*hitl_dispatch_approval_node*)**: Decorated with *@node(rerun_on_resume=True)*, this node checks whether an operator decision is present in *ctx.resume_inputs*. On the first execution pass it yields a *RequestInput* interrupt (*interrupt_id="approve_dispatch"*) and halts the graph; on resume it reads the operator's answer and routes to *approved* or *rejected*.
6. **Terminal action nodes**: *execute_remediation_node* opens a dispatch ticket and allocates a EUR 15 SLA retention credit per High Churn Risk subscriber; *log_rejection_node* suppresses dispatch and records the operator's rejection reason in the audit summary; *monitor_only_node* logs a monitoring ticket without human interruption.
7. **Voice front door (*voice_workflow_dispatcher*)**: A Gemini Live agent configured with a single function tool (*run_pulse_incident_workflow*) that runs the graph inside a nested in-memory session. Because the P1 check lives inside the graph, a voice request for a P1 tower still stops at *PENDING_APPROVAL* and cannot bypass human sign-off.

**Identity mechanics in this lab:** The *persona_id* field inside *IncidentRequest* is a self-asserted string used for workflow attribution and logging. Actual BigQuery queries execute under your local credentials (during notebook runs) or the workflow's Agent Identity (when deployed), and calling *resolve_user_context* on the IAM Context MCP server is informational rather than an IAM enforcement boundary. Production per-user OAuth token propagation is covered in Lab 01's architecture comparison and in Lab 09.

**Experimental API notes:** ADK 2.11.0 marks *ResumabilityConfig* and Gemini Live voice streaming as experimental, so experimental feature warnings in stderr are expected. In Section 10 of *app/agent.py*, the *MultimodalGemini* subclass overrides internal client properties so live voice connections target *LIVE_MODEL* and *LIVE_LOCATION*, verified against pinned *google-adk==2.11.0*.

In [ ]:
%%writefile {WF_AGENT_DIR}/app/agent.py
"""Pulse Telco incident workflow: a typed ADK 2.x graph (parallel branches, join, risk-based routing, human approval)."""
from __future__ import annotations

import contextlib
import hashlib
import json
import logging
import os
import pathlib
import re
from functools import cached_property
from typing import Any

import google.auth
from dotenv import load_dotenv
from google import genai
from google.adk.agents import Agent, LlmAgent
from google.adk.agents.context import Context
from google.adk.apps import App, ResumabilityConfig
from google.adk.events.event import Event
from google.adk.events.request_input import RequestInput
from google.adk.integrations.agent_registry import AgentRegistry
from google.adk.integrations.skill_registry import GCPSkillRegistry
from google.adk.models import Gemini
from google.adk.models.llm_request import LlmRequest
from google.adk.skills import load_skill_from_dir
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.skill_toolset import SkillToolset
from google.adk.workflow import JoinNode, Workflow, node
from google.auth.transport.requests import Request
from google.cloud import bigquery
from google.genai import types
from pydantic import BaseModel, Field, model_validator

load_dotenv()
os.environ.setdefault("GOOGLE_API_USE_CLIENT_CERTIFICATE", "false")

REGION = os.environ.get("REGION", "europe-west4")
MODEL_LOCATION = os.environ.get("MODEL_LOCATION", "eu")
MODEL = os.environ.get("MODEL", "gemini-3.8-flash")
LIVE_MODEL = os.environ.get("PULSE_LIVE_MODEL", "gemini-3.8-live")
LIVE_LOCATION = os.environ.get("PULSE_LIVE_LOCATION") or os.environ.get("LIVE_LOCATION", "eu")
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
# Agent Runtime sets GOOGLE_CLOUD_PROJECT to the project NUMBER; SQL table names need the project ID.
DATA_PROJECT = os.environ.get("PULSE_DATA_PROJECT") or PROJECT_ID
BQ_LOCATION = os.environ.get("BQ_LOCATION", "EU")
BQ_DATASET_CRM = os.environ.get("BQ_DATASET_CRM", "telco_crm")
BQ_DATASET_BILLING = os.environ.get("BQ_DATASET_BILLING", "telco_billing")
BQ_DATASET_NETWORK = os.environ.get("BQ_DATASET_NETWORK", "telco_network")
REGISTRY_LOCATION = os.environ.get("PULSE_REGISTRY_LOCATION") or REGION
SKILLS_LOCATION = os.environ.get("PULSE_SKILLS_LOCATION", "eu")
IAM_MCP_RESOURCE = os.environ.get("PULSE_IAM_MCP_RESOURCE", "")
IAM_MCP_URL = os.environ.get("PULSE_IAM_MCP_URL", "")
SKILLS_DIR = pathlib.Path(__file__).parent / "skills"


# Section 1: Typed schemas for the workflow input and the LLM branch output
class IncidentRequest(BaseModel):
    """Input of the incident workflow. Accepts a typed object, a JSON payload, a Pub/Sub trigger envelope, or chat text."""

    trigger_source: str = Field(default="interactive_chat", description="Who started the run: interactive_chat, live_voice, pubsub_noc_alert, cloud_scheduler_cron, ...")
    region_code: str = Field(default="PRG", description="Region code: PRG, BRN, OVA, BTS, or VIE")
    tower_id: str = Field(default="TWR-PRG-02", description="Target cell tower ID, e.g. TWR-PRG-02 or TWR-BRN-02")
    persona_id: str = Field(default="persona-alex-exec", description="Persona named by the caller (self-asserted in this demo, used for attribution only)")

    @model_validator(mode="before")
    @classmethod
    def _normalize(cls, data: Any) -> Any:
        source = None
        if isinstance(data, str) and data.strip().startswith("{"):
            try:
                data = json.loads(data)
            except json.JSONDecodeError:
                pass
        if isinstance(data, dict) and "data" in data and "attributes" in data:
            # ADK's /trigger/pubsub route hands the agent {"data": <decoded Pub/Sub message>, "attributes": {...}}.
            data, source = data["data"], "pubsub_trigger"
        if isinstance(data, str):
            # Free-text chat message: pick out the region, tower and persona.
            upper = data.upper()
            region = "BRN" if ("BRN" in upper or "BRNO" in upper) else "PRG"
            tower = re.search(r"TWR-[A-Z]{3}-\d{2}", upper)
            data = {
                "region_code": region,
                "tower_id": tower.group(0) if tower else f"TWR-{region}-02",
                "persona_id": "persona-elena-care" if "ELENA" in upper else "persona-alex-exec",
            }
            source = source or "interactive_chat"
        if isinstance(data, dict) and source:
            data = {"trigger_source": source, **data}
        return data


class CrmBillingImpact(BaseModel):
    """Structured output of crm_billing_impact_agent (branch 2)."""

    region_code: str = Field(description="Evaluated region code")
    active_subscribers: int = Field(description="Count of active subscribers in the region")
    high_churn_risk_subscribers: int = Field(description="Count of active subscribers in the region with churn_risk_score >= 0.70")
    delinquent_exposure_eur: float = Field(description="Overdue (>= 30 days) 2026-08 invoice total in EUR for subscribers of this region")
    disputed_exposure_eur: float = Field(description="Disputed 2026-08 invoice total in EUR for subscribers of this region")
    kpi_rule_applied: str = Field(description="Knowledge Catalog / skill KPI rule cited")


# Section 2: Remote MCP servers (BigQuery, Knowledge Catalog, IAM Context via Agent Registry)
_credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])


def _google_auth_headers(_ctx) -> dict[str, str]:
    if not _credentials.valid:
        _credentials.refresh(Request())
    return {"Authorization": f"Bearer {_credentials.token}", "x-goog-user-project": PROJECT_ID}


bigquery_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(url="https://bigquery.googleapis.com/mcp"),
    header_provider=_google_auth_headers,
    tool_filter=["list_dataset_ids", "list_table_ids", "get_table_info", "execute_sql_readonly"],
)

knowledge_catalog_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(url="https://dataplex.googleapis.com/mcp"),
    header_provider=_google_auth_headers,
    tool_filter=["search_entries", "lookup_entry", "lookup_context"],
)

try:
    registry = AgentRegistry(project_id=PROJECT_ID, location=REGISTRY_LOCATION)
    iam_context_tools = registry.get_mcp_toolset(IAM_MCP_RESOURCE)
except Exception:
    iam_context_tools = McpToolset(
        connection_params=StreamableHTTPConnectionParams(url=IAM_MCP_URL),
    )

logging.getLogger("google_adk.google.adk.integrations.skill_registry.gcp_skill_registry").setLevel(logging.ERROR)

local_skills = [load_skill_from_dir(SKILLS_DIR / "bigquery-basics")] if (SKILLS_DIR / "bigquery-basics").exists() else []
analyst_skills = SkillToolset(
    skills=local_skills,
    registry=GCPSkillRegistry(project_id=PROJECT_ID, location=SKILLS_LOCATION),
)


def _jsonable(row) -> dict:
    return {k: (v.isoformat() if hasattr(v, "isoformat") else v) for k, v in dict(row).items()}


# Section 3: Node 1, normalize the trigger (chat, voice, Pub/Sub webhook, cron)
@node
def parse_trigger_payload(ctx: Context, node_input: IncidentRequest) -> Event:
    """Turns any trigger into one IncidentRequest dict plus the prompt for the LLM branch."""
    req = node_input.model_dump() if isinstance(node_input, IncidentRequest) else IncidentRequest.model_validate(node_input).model_dump()
    req["llm_prompt"] = (
        f"Analyze CRM and billing impact for region {req['region_code']} (tower {req['tower_id']}). "
        f"From {DATA_PROJECT}.{BQ_DATASET_CRM}.subscribers count active subscribers and High Churn Risk subscribers "
        f"(churn_risk_score >= 0.70) in region {req['region_code']}. Join {DATA_PROJECT}.{BQ_DATASET_BILLING}.invoices to "
        f"subscribers on subscriber_id, keep region {req['region_code']} and billing_period '2026-08', and sum total_amount_eur "
        f"for delinquent (payment_status='overdue' AND days_past_due>=30) and disputed (payment_status='disputed') invoices."
    )
    return Event(output=req, state={"workflow_trigger": req})


# Section 4: Branch 1 (deterministic node), tower telemetry and open incidents from BigQuery
@node
def tower_telemetry_node(node_input: dict) -> dict:
    """Plain Python + parameterized SQL, no LLM. Runs as the caller's ADC locally and as the Agent Identity when deployed."""
    region = node_input.get("region_code", "PRG")
    tower_id = node_input.get("tower_id", f"TWR-{region}-02")

    bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
    tower_rows = [
        _jsonable(r)
        for r in bq.query(
            f"""
            SELECT tower_id, site_name, region_code, city, radio_tech, band_mhz, avg_utilization_pct, packet_loss_pct, operational_state
            FROM `{DATA_PROJECT}.{BQ_DATASET_NETWORK}.cell_towers`
            WHERE tower_id = @tower_id OR region_code = @region
            ORDER BY CASE WHEN tower_id = @tower_id THEN 0 ELSE 1 END, packet_loss_pct DESC
            LIMIT 1
            """,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[
                    bigquery.ScalarQueryParameter("tower_id", "STRING", tower_id),
                    bigquery.ScalarQueryParameter("region", "STRING", region),
                ]
            ),
        ).result()
    ]
    tower = tower_rows[0] if tower_rows else {"tower_id": tower_id, "region_code": region, "operational_state": "unknown", "packet_loss_pct": 0.0}

    events = [
        _jsonable(r)
        for r in bq.query(
            f"""
            SELECT incident_id, tower_id, severity, incident_status, root_cause_category, affected_subscribers_est, duration_minutes
            FROM `{DATA_PROJECT}.{BQ_DATASET_NETWORK}.network_incidents`
            WHERE tower_id = @tower_id AND incident_status != 'resolved'
            ORDER BY severity ASC
            """,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[bigquery.ScalarQueryParameter("tower_id", "STRING", tower["tower_id"])]
            ),
        ).result()
    ]

    p1_outage = any("P1" in str(e.get("severity", "")) for e in events) or float(tower.get("packet_loss_pct") or 0.0) >= 2.0
    return {
        "persona_id": node_input.get("persona_id", "persona-alex-exec"),
        "trigger_source": node_input.get("trigger_source", "interactive_chat"),
        "tower": tower,
        "open_events": events,
        "is_critical_p1": p1_outage,
    }


# Section 5: Branch 2 (LLM agent node), CRM and billing impact via remote MCP servers and skills
crm_billing_impact_agent = LlmAgent(
    name="crm_billing_impact_agent",
    model=Gemini(
        model=MODEL,
        retry_options=types.HttpRetryOptions(attempts=6, initial_delay=3.0, max_delay=45.0, exp_base=2.0, jitter=1.0, http_status_codes=[429, 500, 502, 503, 504]),
    ),
    description="Quantifies High Churn Risk subscribers and delinquent/disputed billing exposure for one region using BigQuery MCP, Knowledge Catalog MCP and telco skills.",
    instruction=f"""You are the CRM & billing impact node inside Pulse Telco's incident workflow for BigQuery project `{DATA_PROJECT}`.

For the region in the workflow input:
1. If you need the KPI definitions, call `load_skill` with `skill_name="private-pulse-telco-analytics"` or use the Knowledge Catalog MCP. You may call `resolve_user_context` to look up the persona's profile (informational only: BigQuery runs as this agent's own identity).
2. Use `execute_sql_readonly` on the BigQuery MCP toolset:
   - `` `{DATA_PROJECT}.{BQ_DATASET_CRM}.subscribers `` filtered to the target `region_code`: count of `account_status = 'active'` subscribers and count of High Churn Risk subscribers (`account_status = 'active' AND churn_risk_score >= 0.70`).
   - `` `{DATA_PROJECT}.{BQ_DATASET_BILLING}.invoices `` JOINED to `` `{DATA_PROJECT}.{BQ_DATASET_CRM}.subscribers `` ON `subscriber_id`, filtered to the target `region_code` and `billing_period = '2026-08'`: sum of `total_amount_eur` where `payment_status = 'overdue' AND days_past_due >= 30` (delinquent exposure) and sum of `total_amount_eur` where `payment_status = 'disputed'` (disputed exposure).
3. Return the structured `CrmBillingImpact` JSON object.""",
    tools=[iam_context_tools, bigquery_tools, knowledge_catalog_tools, analyst_skills],
    output_schema=CrmBillingImpact,
    output_key="crm_billing_impact",
)


# Section 6: Fan-in (JoinNode) and the risk-based router
merge_diagnostics = JoinNode(name="merge_diagnostics")


@node
def evaluate_severity_router(node_input: dict) -> Event:
    """Approval policy as code: every P1 incident needs a human decision, whatever channel started the run."""
    net = node_input["tower_telemetry_node"]       # JoinNode output = {predecessor_name: output}
    impact = node_input["crm_billing_impact_agent"]
    combined = {
        "trigger_source": net["trigger_source"],
        "persona_id": net["persona_id"],
        "tower": net["tower"],
        "open_events": net["open_events"],
        "is_critical_p1": bool(net.get("is_critical_p1")),
        "crm_billing_impact": impact,
    }
    route = "require_hitl_approval" if combined["is_critical_p1"] else "monitor_only"
    return Event(output=combined, route=route, state={"pending_incident": combined})


# Section 7: Human-in-the-loop approval gate (RequestInput) with approve and reject routes
@node(rerun_on_resume=True)
async def hitl_dispatch_approval_node(ctx: Context, node_input: dict):
    """Pauses the run until an operator answers; routes to 'approved' or 'rejected'."""
    tower = node_input["tower"]
    impact = node_input["crm_billing_impact"]
    prompt_msg = (
        f"[APPROVAL REQUIRED] Tower {tower.get('tower_id')} ({tower.get('city')}) has a P1 outage "
        f"(packet loss {tower.get('packet_loss_pct')}%, {impact.get('high_churn_risk_subscribers')} High Churn Risk subscribers in "
        f"{tower.get('region_code')}), reported via {node_input['trigger_source']}. "
        f"Approve emergency field dispatch and a EUR 15 SLA retention credit? Reply APPROVED or REJECTED: <reason>."
    )
    if not ctx.resume_inputs or "approve_dispatch" not in ctx.resume_inputs:
        yield RequestInput(interrupt_id="approve_dispatch", message=prompt_msg)
        return

    raw_decision = ctx.resume_inputs["approve_dispatch"]
    decision = raw_decision.get("input", str(raw_decision)) if isinstance(raw_decision, dict) else str(raw_decision)
    approved = decision.strip().upper().startswith("APPROV")
    yield Event(
        output={**node_input, "approval_decision": decision},
        route="approved" if approved else "rejected",
        state={"last_approval": decision},
    )


# Section 8: Terminal nodes for dispatch, rejection, and monitor-only outcomes
def _finish(node_input: dict, status: str, action: str, sla_credit_eur: float = 0.0):
    """Builds the final result, yields a readable summary (content) and the structured output."""
    tower = node_input["tower"]
    impact = node_input["crm_billing_impact"]
    tid = tower.get("tower_id", "TWR-PRG-02")
    high_churn = int(impact.get("high_churn_risk_subscribers") or 0)
    ticket_id = "NOC-WF-" + hashlib.sha1(f"{tid}-{node_input['trigger_source']}-{status}".encode()).hexdigest()[:6].upper()
    result = {
        "workflow": "pulse_incident_workflow",
        "status": status,
        "ticket_id": ticket_id,
        "trigger_source": node_input["trigger_source"],
        "persona_id": node_input["persona_id"],
        "tower_id": tid,
        "region_code": tower.get("region_code", "PRG"),
        "tower_status": tower.get("operational_state"),
        "packet_loss_pct": tower.get("packet_loss_pct"),
        "open_events": [e.get("incident_id") for e in node_input.get("open_events", [])],
        "active_subscribers_in_region": impact.get("active_subscribers"),
        "high_churn_risk_subscribers": high_churn,
        "delinquent_exposure_eur": impact.get("delinquent_exposure_eur"),
        "disputed_exposure_eur": impact.get("disputed_exposure_eur"),
        "sla_credit_per_high_churn_sub_eur": sla_credit_eur,
        "total_sla_retention_budget_eur": round(sla_credit_eur * high_churn, 2),
        "approval_decision": node_input.get("approval_decision", "not required (no P1)"),
    }
    summary_text = (
        f"**{status}** (`{ticket_id}`)\n"
        f"- Trigger: `{result['trigger_source']}` (persona `{result['persona_id']}`)\n"
        f"- Network: tower `{tid}` ({result['region_code']}) status=`{result['tower_status']}`, "
        f"packet_loss=`{result['packet_loss_pct']}%`, open incidents={result['open_events']}\n"
        f"- CRM & billing (MCP + skills): {result['active_subscribers_in_region']} active subscribers, "
        f"{high_churn} High Churn Risk, delinquent EUR {result['delinquent_exposure_eur']}, disputed EUR {result['disputed_exposure_eur']}\n"
        f"- Decision: {result['approval_decision']} -> {action}; SLA budget EUR {result['total_sla_retention_budget_eur']}"
    )
    yield Event(
        content=types.Content(role="model", parts=[types.Part.from_text(text=summary_text)]),
        state={"last_workflow_result": result},
    )
    yield Event(output=result)


@node
def execute_remediation_node(node_input: dict):
    """Approved P1: dispatch a field crew and budget the EUR 15 SLA credit per High Churn Risk subscriber."""
    yield from _finish(node_input, "REMEDIATION_DISPATCHED", "field crew dispatched", sla_credit_eur=15.0)


@node
def log_rejection_node(node_input: dict):
    """Rejected P1: no crew, no credits; the operator's reason is kept for the audit trail."""
    yield from _finish(node_input, "DISPATCH_REJECTED", "no dispatch, escalated back to the NOC shift lead")


@node
def monitor_only_node(node_input: dict):
    """No P1: keep watching the tower; nothing to approve."""
    yield from _finish(node_input, "MONITORING", "monitoring ticket opened, no field dispatch")


# Section 9: Assemble the workflow graph
root_agent = Workflow(
    name="pulse_incident_workflow",
    description="5G incident & churn-retention workflow: parallel BigQuery/MCP branches, JoinNode, risk-based routing and human approval.",
    input_schema=IncidentRequest,
    edges=[
        ("START", parse_trigger_payload),
        (parse_trigger_payload, (tower_telemetry_node, crm_billing_impact_agent)),
        ((tower_telemetry_node, crm_billing_impact_agent), merge_diagnostics),
        (merge_diagnostics, evaluate_severity_router),
        (evaluate_severity_router, {"require_hitl_approval": hitl_dispatch_approval_node, "monitor_only": monitor_only_node}),
        (hitl_dispatch_approval_node, {"approved": execute_remediation_node, "rejected": log_rejection_node}),
    ],
)


# Section 10: Voice front door, a Live agent that starts the workflow through a function tool
class MultimodalGemini(Gemini):
    """Demo pattern: routes Live (voice) connections to LIVE_MODEL / LIVE_LOCATION.

    Overrides private ADK members (_live_api_client, _tracking_headers, _live_api_version);
    checked against google-adk 2.11.0, the version this lab pins.
    """

    live_model: str = LIVE_MODEL
    live_location: str = LIVE_LOCATION

    @cached_property
    def _live_api_client(self) -> genai.Client:
        return genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location=self.live_location,
            http_options=types.HttpOptions(
                headers=self._tracking_headers(),
                api_version=self._live_api_version,
            ),
        )

    @contextlib.asynccontextmanager
    async def connect(self, llm_request: LlmRequest):
        llm_request.model = self.live_model
        for content in llm_request.contents or []:
            if content.parts:
                content.parts = [
                    p.model_copy(update={"thought_signature": None})
                    for p in content.parts
                    if not getattr(p, "thought", False)
                ]
        async with super().connect(llm_request) as conn:
            yield conn


async def run_pulse_incident_workflow(
    region_code: str = "PRG",
    tower_id: str = "TWR-PRG-02",
    persona_id: str = "persona-alex-exec",
) -> dict:
    """Runs the Pulse incident workflow for one tower and returns the outcome (P1 incidents come back as PENDING_APPROVAL)."""
    from google.adk.runners import InMemoryRunner

    runner = InMemoryRunner(app=app)  # nested runner: its own in-memory session, not linked to the voice session
    session = await runner.session_service.create_session(app_name=app.name, user_id=persona_id)
    payload = {"trigger_source": "live_voice", "region_code": region_code, "tower_id": tower_id, "persona_id": persona_id}
    msg = types.Content(role="user", parts=[types.Part.from_text(text=json.dumps(payload))])
    outcome: dict = {"status": "UNKNOWN"}
    async for ev in runner.run_async(user_id=persona_id, session_id=session.id, new_message=msg):
        if isinstance(ev.output, dict) and "status" in ev.output:
            outcome = ev.output
        for part in (ev.content.parts if ev.content and ev.content.parts else []):
            if part.function_call and part.function_call.name == "adk_request_input":
                outcome = {"status": "PENDING_APPROVAL", "approval_request": (part.function_call.args or {}).get("message")}
    return outcome


voice_workflow_dispatcher = Agent(
    name="pulse_voice_workflow_dispatcher",
    model=MultimodalGemini(model=MODEL),
    description="Voice dispatcher that starts the Pulse incident workflow.",
    instruction="""You are Pulse Telco's voice operations dispatcher.
When the operator asks you to run the incident workflow for a region or tower:
1. Call `run_pulse_incident_workflow` with the requested `region_code`, `tower_id`, and `persona_id`.
2. Answer in 2-3 short spoken sentences, no markdown:
   - status PENDING_APPROVAL: say it is a P1 incident, so the field dispatch waits for an operator's approval; mention packet loss or affected subscribers if given.
   - status MONITORING: say no dispatch is needed and the tower stays under monitoring.
   - any other status: read the status and ticket ID.""",
    tools=[run_pulse_incident_workflow],
)

app = App(
    name="app",
    root_agent=root_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

## Step 8.3 (Mode 1): Interactive Chat Trigger, Human Approval Pause, and Resume

When a workflow pauses for human approval, where does the execution state live while the operator is reading the alert?

In **Run 1 (Pause)** below, we trigger the workflow with a chat message targeting Prague tower *TWR-PRG-02*, which has an active **P1** fiber-cut outage (*INC-7001*). After trigger normalization, both diagnostic branches run in parallel, *JoinNode* merges their outputs, and the severity router selects *require_hitl_approval*. When the approval node yields *RequestInput*, ADK serializes the pending interrupt into the session history as a synthetic function call named *adk_request_input* and terminates the generator cleanly.

In **Run 2 (Resume)**, the operator's decision is delivered back to *runner.run_async* as a *FunctionResponse* for *adk_request_input* (matching *id="approve_dispatch"* and the paused *invocation_id*). That response payload is the exact wire contract an external Microsoft Teams adaptive card or NOC web console sends when an engineer clicks Approve or Reject.

**Durability comparison:** Locally, *InMemoryRunner* stores the paused session in Python process memory, which disappears if you restart the Jupyter kernel. When deployed to Agent Runtime (Step 8.6), sessions are persisted in the managed *VertexAiSessionService*, so a paused P1 workflow can wait hours for a shift lead's response without holding a container thread open. That gives you the durability of n8n's database-backed Wait node without managing your own PostgreSQL and Redis cluster.

The *start_incident()* and *resume_incident()* helpers defined below are reused in the hands-on exercises that follow.

In [ ]:
# Step 8.3 (Mode 1): Chat trigger, parallel branches, pause at the approval gate, and resume
sys.path.insert(0, str(WF_AGENT_DIR))
for m in [k for k in list(sys.modules) if k == "app" or k.startswith("app.")]:
    del sys.modules[m]          # re-import after every edit of agent.py

from app.agent import app as workflow_app, voice_workflow_dispatcher
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

wf_runner = InMemoryRunner(app=workflow_app)

def show_wf_event(ev):
    """show_event() plus the graph part: which route a node chose."""
    show_event(ev)
    route = getattr(ev.actions, "route", None)
    if route:
        print(f"  => [{ev.author}] route: {route}")

async def start_incident(text, user_id="persona-alex-exec"):
    """Starts the workflow from a chat message. Returns (session_id, invocation_id, approval request or None)."""
    session = await wf_runner.session_service.create_session(app_name=workflow_app.name, user_id=user_id)
    msg = genai_types.Content(role="user", parts=[genai_types.Part.from_text(text=text)])
    invocation_id, approval_request = None, None
    async for ev in wf_runner.run_async(user_id=user_id, session_id=session.id, new_message=msg):
        invocation_id = ev.invocation_id
        show_wf_event(ev)
        for part in (ev.content.parts if ev.content and ev.content.parts else []):
            if part.function_call and part.function_call.name == "adk_request_input":
                approval_request = (part.function_call.args or {}).get("message")
    return session.id, invocation_id, approval_request

async def resume_incident(session_id, invocation_id, decision, user_id="persona-alex-exec"):
    """Sends the operator's answer to the paused run (the message a Teams card or web form would send)."""
    msg = genai_types.Content(role="user", parts=[genai_types.Part(function_response=genai_types.FunctionResponse(
        id="approve_dispatch", name="adk_request_input", response={"input": decision}))])
    async for ev in wf_runner.run_async(user_id=user_id, session_id=session_id, invocation_id=invocation_id, new_message=msg):
        show_wf_event(ev)

print("=== Run 1: chat trigger -> parallel branches -> pause at the approval gate ===")
sid, inv, request = await start_incident("Run incident workflow for PRG tower TWR-PRG-02 as persona-alex-exec")
print(f"\n⏸️  PAUSED (invocation_id={inv}):\n  {request}\n")

print("=== Run 2: the operator approves -> the same invocation resumes ===")
await resume_incident(sid, inv, "APPROVED by Alex Kral: dispatch the Prague RAN crew and apply the EUR 15 SLA credit")

### 🧪 Try It Yourself (5 Minutes Each)
1. **Exercise the rejection branch.** Start a fresh P1 workflow run for Prague and resume it with a rejection rationale:
```python
sid, inv, _ = await start_incident("Run incident workflow for PRG tower TWR-PRG-02 as persona-alex-exec")
await resume_incident(sid, inv, "REJECTED: no crew available before 18:00")
```
Verify in the event output that the approval node selects the *rejected* route, final status is *DISPATCH_REJECTED*, the total SLA retention budget remains EUR 0, and your rejection reason is preserved in *approval_decision*.
2. **Tighten the approval policy in code.** Because the routing logic lives in *evaluate_severity_router* (inside the *%%writefile* cell in Step 8.2), you can require human sign-off whenever a region has at least 5 High Churn Risk subscribers, even during a lower-severity outage:
```python
route = "require_hitl_approval" if combined["is_critical_p1"] or impact.get("high_churn_risk_subscribers", 0) >= 5 else "monitor_only"
```
Update that line, re-run Step 8.2 and Step 8.3, and test Brno:
```python
await start_incident("Run incident workflow for BRN tower TWR-BRN-02")
```
Brno tower *TWR-BRN-02* has a P2 incident rather than a P1, but Brno has 5 High Churn Risk subscribers, so the workflow now pauses for operator approval. Because the policy is Python code, changes go through standard pull-request review and unit testing.

## Step 8.4 (Mode 2): Voice Front Door with Gemini Live

Field engineers and NOC shift leads often work hands-free during an outage, making voice a natural interface for kicking off diagnostics. Architecturally, however, you never want a separate "voice workflow" with duplicated business logic that can drift out of sync with your text or webhook workflows.

In our design, *voice_workflow_dispatcher* is a lightweight front-door agent running on **gemini-3.8-live** over the *eu* multi-region endpoint. It exposes a single function tool (*run_pulse_incident_workflow*), which executes the incident workflow graph inside a nested runner and returns the structured outcome to the voice model:
* **Simulated utterance input**: The cell sends a user turn into the bidirectional stream via *live_queue.send_content(...)*. In a browser or mobile client, you stream raw 16 kHz PCM microphone audio chunks via *live_queue.send_realtime(...)*.
* **Spoken audio output**: Gemini Live streams back 24 kHz PCM audio frames alongside an output transcription, rendered in an inline audio player below the cell.
* **Channel-invariant governance**: Because *TWR-PRG-02* has a P1 outage, the underlying graph pauses at *RequestInput* and returns *PENDING_APPROVAL* to the tool. The voice dispatcher informs the caller over audio that field dispatch is waiting for operator sign-off.
* **Stream termination**: The loop exits with *break* as soon as the spoken turn completes, which causes ADK to log a normal live-run cancellation message.

⏱ About 1 minute, as the function tool executes both parallel diagnostic branches before the voice model speaks the summary.

In [ ]:
# Step 8.4 (Mode 2): Voice front door (Live model) calling the workflow through a function tool
import asyncio, io, wave
from IPython.display import Audio, display
from google.adk.agents.live_request_queue import LiveRequestQueue
from google.adk.agents.run_config import RunConfig, StreamingMode

voice_runner = InMemoryRunner(agent=voice_workflow_dispatcher, app_name="app")
voice_session = await voice_runner.session_service.create_session(app_name="app", user_id="persona-alex-exec")

live_queue = LiveRequestQueue()
live_queue.send_content(genai_types.Content(role="user", parts=[genai_types.Part.from_text(
    text="Run the Pulse incident workflow for Prague region PRG and tower TWR-PRG-02 as persona-alex-exec.")]))

pcm_audio, partial_text, final_text = bytearray(), [], []
print("=== Mode 2: voice front door (Live model -> run_pulse_incident_workflow) ===")
try:
    async with asyncio.timeout(120):
        async for ev in voice_runner.run_live(
            user_id="persona-alex-exec",
            session_id=voice_session.id,
            live_request_queue=live_queue,
            run_config=RunConfig(
                streaming_mode=StreamingMode.BIDI,
                response_modalities=["AUDIO"],
                output_audio_transcription=genai_types.AudioTranscriptionConfig(),
            ),
        ):
            show_event(ev)                           # the tool call and the workflow outcome (tool result)
            for p in (ev.content.parts if ev.content and ev.content.parts else []):
                if p.inline_data and p.inline_data.data:
                    pcm_audio.extend(p.inline_data.data)
            if ev.output_transcription and ev.output_transcription.text:
                (final_text if ev.output_transcription.finished else partial_text).append(ev.output_transcription.text)
            if ev.turn_complete and pcm_audio:
                break                                # stop listening after the spoken answer
except TimeoutError:
    print("(timeout: showing what arrived so far)")
finally:
    live_queue.close()

transcript = " ".join(final_text) or "".join(partial_text)
print(f"\n🎙️ Spoken answer ({len(pcm_audio):,} bytes of 24 kHz PCM):\n  {transcript or '(no transcription received)'}")
if pcm_audio:
    wav_buf = io.BytesIO()
    with wave.open(wav_buf, "wb") as wav:
        wav.setnchannels(1); wav.setsampwidth(2); wav.setframerate(24000)
        wav.writeframes(bytes(pcm_audio))
    display(Audio(wav_buf.getvalue(), rate=24000, autoplay=False))

## Step 8.5 (Modes 3 and 4): Event-Driven Pub/Sub Webhooks and Scheduled Cron Runs

Most operational workflows are not started by a human typing in a chat window; they are triggered by an upstream monitoring alarm or a recurring schedule. In n8n, you wire a Webhook node or a Schedule Trigger node to the start of the canvas.

ADK provides native **ambient trigger endpoints** for event-driven execution. Passing *trigger_sources=["pubsub"]* to *get_fast_api_app(...)* (or starting the server with *adk api_server --trigger_sources pubsub*) mounts the *POST /apps/{app_name}/trigger/pubsub* endpoint, which handles the cloud plumbing automatically:
1. **Envelope decoding**: Parses the standard Cloud Pub/Sub push JSON envelope and base64-decodes *message.data* into the payload passed to *parse_trigger_payload*.
2. **Session isolation**: Creates a dedicated session per message (deriving the user ID from the subscription path) and runs the root workflow.
3. **Backpressure and retry**: Caps concurrent in-flight runs per container instance via *ADK_TRIGGER_MAX_CONCURRENT* (default 10) and retries transient HTTP 429 errors with exponential backoff via *ADK_TRIGGER_MAX_RETRIES* (default 3). Returning a non-2xx HTTP status signals Pub/Sub to redeliver the message.

The code cell below verifies both event-driven modes locally without provisioning external topics:
* Updates the scaffolded *app/fast_api_app.py* to include *trigger_sources=["pubsub"]* so the container exposes the push endpoint whenever deployed to Cloud Run.
* Boots the ADK FastAPI server **in-process** using Starlette/FastAPI's *TestClient* and posts two realistic Pub/Sub push envelopes:
  * **Mode 3 (Event-driven NOC webhook)**: An anomaly detector alert for *TWR-PRG-02* (P1). Even though no human initiated the request, the risk router still pauses the workflow at *adk_request_input*.
  * **Mode 4 (Scheduled cron sweep)**: An hourly RAN health sweep for Brno tower *TWR-BRN-02* (P2), which flows straight through to the *MONITORING* terminal state. In production, Cloud Scheduler publishes this JSON payload to the Pub/Sub topic on a cron schedule, and the push subscription delivers it to the exact same endpoint.
* Queries the server's session inspection API to print the recorded event trajectory for each run.

**Production hosting topology:** ADK's documentation recommends **Cloud Run** as the primary runtime for ambient agents using HTTP push triggers (deployed from the same project via *agents-cli deploy --deployment-target cloud_run*). If you host the workflow on Agent Runtime (Step 8.6), place a lightweight Cloud Run service or Cloud Run function in front of it to receive the Pub/Sub push and forward the payload to the Agent Runtime query API. Similarly, when an unmanned Mode 3 run pauses at *RequestInput*, your notification handler posts an adaptive card to Microsoft Teams or email whose Approve/Reject buttons invoke the session resume API.

Reference production commands for wiring Cloud Pub/Sub and Cloud Scheduler to a Cloud Run deployment (not executed in this lab; set *RUN_URL* to your Cloud Run service URL and *PUSH_SA* to a service account granted *roles/run.invoker*):
```bash
gcloud pubsub topics create noc-alerts --project="$PROJECT_ID"
gcloud pubsub subscriptions create noc-alerts-push --project="$PROJECT_ID" --topic=noc-alerts \
  --push-endpoint="$RUN_URL/apps/app/trigger/pubsub" --push-auth-service-account="$PUSH_SA"
gcloud scheduler jobs create pubsub hourly-brno-ran-sweep --project="$PROJECT_ID" --location="$REGION" \
  --schedule="0 * * * *" --topic=noc-alerts \
  --message-body='{"trigger_source": "cloud_scheduler_cron", "region_code": "BRN", "tower_id": "TWR-BRN-02"}'
```

In [ ]:
# Step 8.5 (Modes 3 and 4): Send Pub/Sub push envelopes to ADK's /trigger/pubsub route in-process
import base64
from fastapi.testclient import TestClient
from google.adk.cli.fast_api import get_fast_api_app

# (a) Make the scaffold's server trigger-ready (used when this project is served from Cloud Run)
fast_api_path = WF_AGENT_DIR / "app" / "fast_api_app.py"
fa_txt = fast_api_path.read_text()
if "trigger_sources" not in fa_txt and "web=True," in fa_txt:
    fast_api_path.write_text(fa_txt.replace("web=True,", 'web=True,\n    trigger_sources=["pubsub"],', 1))
    print("Added trigger_sources=['pubsub'] to app/fast_api_app.py")
print("app/fast_api_app.py has the Pub/Sub route enabled:", "trigger_sources" in fast_api_path.read_text())

# (b) The same ADK API server, in-process, with in-memory sessions
trigger_api = get_fast_api_app(agents_dir=str(WF_AGENT_DIR), web=False, trigger_sources=["pubsub"],
                               session_service_uri="memory://", artifact_service_uri="memory://", memory_service_uri="memory://")

def push_pubsub(payload, subscription):
    """POSTs a Pub/Sub push envelope to the trigger route, then reads back the session the route created."""
    envelope = {
        "message": {"data": base64.b64encode(json.dumps(payload).encode()).decode(), "messageId": payload.get("event_id", "manual-1")},
        "subscription": subscription,
    }
    with TestClient(trigger_api) as client:
        r = client.post("/apps/app/trigger/pubsub", json=envelope)
        print(f"POST /apps/app/trigger/pubsub -> HTTP {r.status_code} {r.text[:300]}")
        if r.status_code != 200:
            raise RuntimeError("trigger route failed - see the response above")
        user_id = subscription.replace("/", "--")      # the route derives the user ID from the subscription name
        sessions = client.get(f"/apps/app/users/{user_id}/sessions").json()
        latest = max(sessions, key=lambda s: s.get("lastUpdateTime") or 0)
        session = client.get(f"/apps/app/users/{user_id}/sessions/{latest['id']}").json()
    for ev in session.get("events", []):
        show_event(ev)
    result = session.get("state", {}).get("last_workflow_result")
    print("->", result["status"] if result else "PAUSED: waiting for an operator decision (adk_request_input)", "\n")
    return session

print("\n=== MODE 3: Pub/Sub push from the NOC anomaly detector (TWR-PRG-02, P1) ===")
mode3 = push_pubsub({"trigger_source": "pubsub_noc_alert", "event_id": "EVT-9001", "region_code": "PRG",
                     "tower_id": "TWR-PRG-02", "persona_id": "persona-alex-exec"},
                    f"projects/{PROJECT_ID}/subscriptions/noc-alerts")

print("=== MODE 4: Cloud Scheduler cron -> Pub/Sub (hourly RAN sweep, TWR-BRN-02) ===")
mode4 = push_pubsub({"trigger_source": "cloud_scheduler_cron", "schedule_name": "hourly-brno-ran-sweep",
                     "region_code": "BRN", "tower_id": "TWR-BRN-02", "persona_id": "persona-alex-exec"},
                    f"projects/{PROJECT_ID}/subscriptions/ran-sweep")

### 🧪 Try It Yourself (5 Minutes Each)
1. **Configure per-node retry policies in ADK.** In n8n, you enable retries in a node's settings dialog. In ADK, node resilience is declared directly on the *@node* decorator. In Step 8.2's code cell, import *RetryConfig* from *google.adk.workflow* and update the telemetry node decorator:
```python
from google.adk.workflow import JoinNode, RetryConfig, Workflow, node

@node(retry_config=RetryConfig(max_attempts=3), timeout=20)
def tower_telemetry_node(node_input: dict) -> dict:
```
Re-run Step 8.2, Step 8.3, and Step 8.5 to verify that any transient BigQuery network hiccup on that node is automatically retried up to three times within a 20-second timeout before failing the workflow.
2. **Test an Ostrava NOC alert.** Execute a Pub/Sub push for Ostrava tower *TWR-OVA-02* and inspect which branch the router chooses:
```python
push_pubsub(
    {"trigger_source": "pubsub_noc_alert", "region_code": "OVA", "tower_id": "TWR-OVA-02"},
    f"projects/{PROJECT_ID}/subscriptions/noc-alerts",
)
```
Check the open incident severity and packet-loss percentage returned for *TWR-OVA-02* to see why it takes that route.

## Step 8.6: *(Optional)* Deploy the Workflow Agent to Agent Runtime

In the final step of this lab, we deploy *pulse-workflow-agent* to **Agent Runtime** in the workshop region (*REGION*) with its own dedicated **Agent Identity**, and invoke it in the cloud using the Mode 3 NOC alert payload.

Execution is controlled by two flags at the top of the cell:
* *DEPLOY_WORKFLOW_AGENT = True*: Set this to *False* if you want to skip the 5 to 10 minute container build during a time-boxed session.
* *FORCE_REDEPLOY = False*: Matches the idempotent deployment guard from Lab 02; set to *True* when you want to push local code edits in *app/agent.py* to an already deployed instance.

What the deployment sequence does under the hood:
1. Runs *agents-cli deploy*, which builds the container image, provisions the Agent Runtime instance, and writes the cloud resource identifier to *pulse-workflow-agent/deployment_metadata.json*.
2. Queries the Agent Runtime REST API to read the instance's *effectiveIdentity* (*principal://...*) and grants it least-privilege project roles: BigQuery Data Viewer, BigQuery Job User, Knowledge Catalog (Dataplex) Catalog Viewer, MCP Tool User, Service Usage Consumer, and Agent Registry Viewer, followed by a 60-second wait for IAM policy propagation.
3. Streams the P1 NOC alert payload to the cloud instance. Just as in the local test, execution pauses at the approval gate and emits the *adk_request_input* event, now persisted inside the managed cloud session store.

**API naming reminder:** **Agent Runtime** is the platform capability name, *reasoningEngines/...* is the underlying REST resource collection, and *vertexai.Client(...).agent_engines* is the Python SDK accessor (a *FutureWarning* from *vertexai.Client* is expected in current SDK releases).

Equivalent terminal command:
```bash
cd pulse-workflow-agent
agents-cli deploy --project "$PROJECT_ID" --region "$REGION" --agent-identity --no-confirm-project
```

In [ ]:
# Step 8.6 (Optional): Deploy pulse-workflow-agent to Agent Runtime, grant IAM roles, and invoke it
import time, requests, google.auth, vertexai
from google.auth.transport.requests import Request

DEPLOY_WORKFLOW_AGENT = True    # set False to skip the 5-10 min deploy
FORCE_REDEPLOY = False          # set True to push code changes to an existing deployment

async def deploy_and_call_workflow_agent():
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())

    meta_file = WF_AGENT_DIR / "deployment_metadata.json"
    already_deployed = False
    if meta_file.exists():
        meta_txt = meta_file.read_text()
        if f"projects/{PROJECT_NUMBER}/" in meta_txt or f"projects/{PROJECT_ID}/" in meta_txt:
            rid = json.loads(meta_txt).get("remote_agent_runtime_id") or ""
            chk = requests.get(f"https://{REGION}-aiplatform.googleapis.com/v1/{rid}", headers={"Authorization": f"Bearer {creds.token}"})
            already_deployed = chk.status_code == 200 and bool(chk.json().get("spec", {}).get("effectiveIdentity"))

    deploy_cmd = f"agents-cli deploy --project {PROJECT_ID} --region {REGION} --agent-identity --no-confirm-project"
    terminal(deploy_cmd, cwd=WF_AGENT_DIR)
    if already_deployed and not FORCE_REDEPLOY:
        print("pulse-workflow-agent is already deployed - set FORCE_REDEPLOY = True to push changes.")
    else:
        r = subprocess.run(deploy_cmd, cwd=WF_AGENT_DIR, shell=True, text=True, capture_output=True)
        print(r.stdout[-2500:])
        if r.returncode != 0:
            print(r.stderr[-2500:])
            raise RuntimeError("workflow deploy failed")

    meta = json.loads(meta_file.read_text())
    wf_engine = meta.get("remote_agent_runtime_id") or meta.get("remote_agent_engine_id")
    wf_url = f"https://{REGION}-aiplatform.googleapis.com/v1/{wf_engine}"
    creds.refresh(Request())
    info = requests.get(wf_url, headers={"Authorization": f"Bearer {creds.token}"}).json()
    raw_id = info.get("spec", {}).get("effectiveIdentity") or info.get("identity", "")
    wf_principal = raw_id if raw_id.startswith("principal://") else f"principal://{raw_id}"
    save_to_workshop_env(
        PULSE_WORKFLOW_AGENT_ENGINE=wf_engine,
        PULSE_WORKFLOW_AGENT_ENGINE_ID=wf_engine.split("/")[-1],
        PULSE_WORKFLOW_AGENT_URL=wf_url,
        PULSE_WORKFLOW_AGENT_PRINCIPAL=wf_principal,
    )
    print("Workflow Agent Identity:", wf_principal)

    for role in ["roles/bigquery.dataViewer", "roles/bigquery.jobUser", "roles/dataplex.catalogViewer",
                 "roles/mcp.toolUser", "roles/serviceusage.serviceUsageConsumer", "roles/agentregistry.viewer"]:
        for attempt in range(4):
            res = subprocess.run(f'gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{wf_principal}" '
                                 f'--role={role} --condition=None --quiet >/dev/null', shell=True)
            if res.returncode == 0:
                break
            time.sleep(2 * (attempt + 1))
        else:
            print(f"WARNING: could not grant {role} to {wf_principal} - check your Project IAM Admin role")
    print("Granted least-privilege IAM roles to the workflow's Agent Identity.")
    if not already_deployed:
        print("waiting 60 s for the new IAM bindings to propagate before the first cloud call...")
        time.sleep(60)

    wf_client = vertexai.Client(project=PROJECT_ID, location=REGION)
    remote_wf = wf_client.agent_engines.get(name=wf_engine)
    session_id = (await remote_wf.async_create_session(user_id="persona-alex-exec"))["id"]
    payload = json.dumps({"trigger_source": "pubsub_noc_alert", "event_id": "EVT-9001", "region_code": "PRG",
                          "tower_id": "TWR-PRG-02", "persona_id": "persona-alex-exec"})
    print(f"\nCalling the deployed pulse-workflow-agent in {REGION} (session={session_id}):")
    stats = {}
    async for ev in remote_wf.async_stream_query(user_id="persona-alex-exec", session_id=session_id, message=payload):
        show_event(ev, stats)
    show_stats(stats)

if DEPLOY_WORKFLOW_AGENT:
    await deploy_and_call_workflow_agent()
else:
    print("Skipped: set DEPLOY_WORKFLOW_AGENT = True to deploy.")

## ✅ Lab 08 Complete: Architecture Recap

* **Typed code graphs vs visual canvases:** *pulse_incident_workflow* combines deterministic *@node* Python functions with an *LlmAgent* node, executes both diagnostic branches concurrently, synchronizes their typed outputs via *JoinNode*, and enforces routing policies in code.
* **Channel-invariant human approval:** Every P1 outage pauses at *RequestInput* regardless of whether the trigger arrived via interactive chat, Gemini Live voice, or an automated Pub/Sub push webhook. Sending back the operator's *FunctionResponse* resumes the paused invocation: *APPROVED* dispatches the field crew and budgets SLA credits, while *REJECTED* halts dispatch and preserves the rejection rationale.
* **Multi-channel trigger mechanics:** Chat and voice invoke the workflow graph directly, while event-driven webhooks and scheduled cron sweeps target ADK's built-in */apps/app/trigger/pubsub* endpoint using standard Pub/Sub push envelopes.
* **Shared platform foundation:** The workflow reuses the exact same governed building blocks established in Labs 03 to 06: the managed BigQuery and Knowledge Catalog MCP servers, the custom IAM Context MCP server on Cloud Run, and Skill Registry skills.
* **Pragmatic n8n coexistence:** Keep n8n where turnkey SaaS connectors and built-in approval cards shine, use ADK on GEAP where concurrent LLM graphs, dedicated Agent Identities, and automated evaluations matter, and bridge the two over HTTP/A2A and MCP.
* **Managed cloud deployment:** The workflow agent can be deployed to Agent Runtime with its own least-privilege Agent Identity.

👉 **Next steps:**
* *(Self-Study)* **[Lab 09: Network-Enforced Tool Governance with Agent Gateway](lab09_agent_gateway.ipynb)**
* *(Self-Study)* **[Lab 10: Publishing Agents to Gemini Enterprise](lab10_gemini_enterprise.ipynb)**
* **[Lab 11: Environment Teardown and Workshop Architecture Recap](lab11_cleanup.ipynb)**